# Tissue extractor tutorial

Tutorial for running the tissue extractor.

# should rename this - this isn't the whole tutorial.

Make this file the file for processing (norm - call once, then convert cell to `raw`; reslice - for all images) for slope, intercept images.

## to do for tutorial

- make the base loop a helper function that you can call for each function

- run each function in its own cell using the helper function

- Above is for the multiple subject case

- Also just show how to do it on one subject

In [1]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
import ants

import SUITPy as suit
import SUITPy.atlas as atlas

import nitools as nt
import tissue_extractor as te

from pathlib import Path
import os

# BEFORE RUNNING: UPLOAD TO GITHUB REPO (THIS AND TISSUE EXTRACTOR, UPDATED); THEN RUN

In [2]:
"""
dummy function to test loop; to be replaced with the actual function
"""
def dummy_fcn(subj_id, week):
    print(subj_id, week)

In [2]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [3]:
tissue = 'wm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

In [6]:
te.normalize?

Signature: te.normalize(t1_path, mask_path, results_path, space='SUIT')
Docstring:
June 9: updated with updated suit function
with correct input arg names
and option to choose template space to normalize to
File:      ~/Documents/GitHub/smarts_cerebellum/image_processing/tissue_extractor.py
Type:      function

# ask joern
before running this!
In the regression module, should I put it back to voxel coordinates before writing image (intercept and slope)?

Also should talk to Joern about the voxel reg on just the t1 anatomical.

# check the code for the new week_path before running

# restart kernel for updated av function before calling this

figure out why it cannot find image_analysis folder; as a temporary solution, just write the function here.

In [6]:
from image_analysis import avg_vol as av

ModuleNotFoundError: No module named 'image_analysis'

In [ ]:

def avg_vol(subj_id, 
            reference_img, # reference anatomical
            #week_path, # path to week image
            results_path,
            image_suffix,
            tissue=None):
    """
    Inputs:
    anat dir: participants file OR [(subj, week) and call it inside a loop].
    Reference img (inside the Jupyter notebook loop for reading off the info file)
    #week_path (path for each week's image), results_path (store results)
    results path: directory to store results
    image suffix: suffix with which to save the slope and intercept images
        suggested: <image_type>_<space> where 'image_type' is "anat", "wm", "gm", etc; 'space' is native or template (<template_name>)

    Everything is done in the reference image. So this function will (...) (resample voxels in other weeks so that they are aligned with the reference, and perform multiple linear regression)

    Returns B_hat coefficient matrix (for more flexibility in other possible operations)
    
    """

    # file prefix encoding (based on SPM segmentation notation)
    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c2'
    }

    img0 = nib.load(reference_img)

    # later fix: option to reduce to only wtihin-brain voxels
    
    # transform into world coordinates
    i, j, k = np.indices(img0.shape) # matrix indices for premult by affine
    x,y,z = nt.affine_transform(i, j, k, img0.affine)

    # all possible weeks.
    weeks = np.array([0,4,12,24,52]) # read from file, use file reading function maybe
    
    
    # THIS PART SHOULD BE DONE IN TUTORIAL, NOT IN FUNCTION. or with a helper function.


    #____________________________________
    # find the number of measurement weeks that exist
    p_weeks = []
    for week in weeks:
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        #week_path = f'{anat_dir}/{subj_id}/W{week}/c2{subj_id}_W{week}_T1.nii' # fix

        if not tissue==None:
            week_path = f'{anat_dir}/{subj_id}/W{week}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        else:
            week_path = f'{anat_dir}/{subj_id}/W{week}/{subj_id}_W{week}_T1.nii'


        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        p_weeks.append(week)
    
    if len(p_weeks) == 1: # only one measurement week available
        return None # exit function (skip subject)    
    #__________________________


    Y = np.zeros((len(p_weeks), np.prod(img0.shape))) # initialize Y (shape = (k by p)) array, where k = number of weeks available

    for week in weeks: # resample ALL weeks, including the reference week
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        
        if not tissue==None:
            week_path = f'{anat_dir}/{subj_id}/W{week}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        else:
            week_path = f'{anat_dir}/{subj_id}/W{week}/{subj_id}_W{week}_T1.nii'


        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        week_img = nib.load(week_path)

        week_dict = {
            '0': 0,
            '4': 1,
            '12': 2,
            '24': 3,
            '52': 4
        }

    
        # resample each week's image so that voxels are exactly on top of reference week voxels; add to response matrix as row vector
        Y[week_dict[str(week)]:,] = nt.sample_image(week_img, # response matrix
                                xm=x, ym = y, zm = z, # world coordinates
                                interpolation = 1 # using trilinear resampling
                                ).flatten() # need to put each week as a row
        
        # now we have Y as a k by p matrix, where k is the number of weeks.  

    # design matrix
    num_weeks = len(p_weeks)
    X = [np.ones(shape = (num_weeks)), p_weeks]
    X = np.array(X)
    X = X.T

    # estimator (coefficients matrix)
    B_hat = np.linalg.pinv(X) @ Y
    # where B_hat = [B_0 B_1].T

    # intercept and slope reshaped into Nifti-compatible array
    intercept = B_hat[0,:].reshape(img0.shape)
    slope = B_hat[1,:].reshape(img0.shape)

    """
    So this image is in world coordinates now, and saved with the affine of the original image.
    Should it be converted back to voxel coordinates (bc otherwise, the affine is kinda meaningless)?
    """
    

    """
    EDITS
    """

    # back into voxel coordinates
    #iv, jv, zv = np.indices()


    # save as Nifti
    intercept_img = nib.Nifti1Image(intercept, img0.affine)
    slope_img = nib.Nifti1Image(slope, img0.affine)

    # fix: name of file should be insertable, too (e.g. which_type = 'native' or smth in fcn input)
    nib.save(intercept_img, f'{results_path}/{subj_id}_T1_intercept_{image_suffix}.nii.gz') # specify file name
    nib.save(slope_img, f'{results_path}/{subj_id}_T1_slope_{image_suffix}.nii.gz')

    return B_hat


SyntaxError: incomplete input (784224051.py, line 124)

In [9]:
subj_id = 'CU_2538'
week = 'W0'

results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/' # folder specifies space

t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'


In [15]:
p_df=p_df[p_df['subj_id']=='CU_2538']

In [16]:
p_df

,SN,ID,Centre,Week,week,RefT1,numrun,nslices,Hand,LesionSide,...,surfmvpa,include1,lesiondef,behavior_missing,behavior_blocks,has_mvc,DTImap_missing,CentreNo,machine,subj_id
5,6,2538,CU,W0,0,W0,8,35,b,right,...,1,1,1,0,8,1,0,1,naveed,CU_2538
6,7,2538,CU,W4,4,W0,8,35,b,right,...,1,1,1,0,8,1,0,1,naveed,CU_2538


In [ ]:
# CALL FOR WM SEGMENTATION IMAGE IN NATIVE SPACE________change image suffix for other types

betas = [] # store matrices for all subjects

for subj in p_df['subj_id'].unique():
    #betas.append(avg_vol(subj, ref_img))
    refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
    ref_img = f'{anat_dir}/{subj}/{refT1}/c2{subj}_{refT1}_T1.nii' # should specify tissue in this call for reference iamge

    avg_vol(subj_id, 
            reference_img=ref_img, # reference anatomical
            #week_path, # path to week image
            results_path = f'{anat_dir}/{subj}/', # maybe specify folder for this, too (e.g. native_regression)
            image_suffix = 'wm_native',
            tissue='wm')
    
"""
    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,
                         results_path = f'{anat_dir}/{subj}/'),
                         image_suffix = "wm_native"
                         )
                         """
    
    # we don't need this weeks loop, the function does it. Really just need to loop through the subject and get their reference img
    # even better if the function can get the reference image itself.
    
    #week = p_df.loc[(p_df['subj_id']==subj), 'week'].iloc[1]
   



'\n    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,\n                         results_path = f\'{anat_dir}/{subj}/\'),\n                         image_suffix = "wm_native"\n                         )\n                         '

# note
In the present avg_vol function, week paths are for c2; this is terrible, need to fix (i.e. have a tissue_dict, where if tissue = None, then it'll just put nothing in front, so like):

tissue_dict = {
    'gm': c1,
    'wm': c2,
    'csf': c3,
    None: '' # i don't know if this notation would work
}

Otherwise, we can just have an if tissue!=None statement, so if tissue is supplied, it'll use the dict above (minus None coding, which is strange and might not work) to find the correct prefix of the file.

I did the latter.

Should check that this works well on one subject before continuing wiht the rest. ANd need to ask Joern about the voxel coordiantes thing first, but also check the code from nilearn.